# 04 — Score Generation and Fairness Evaluation

## Overview

This notebook is the final analytical stage of **Image Compression Effects on Face Recognition Fairness**.

It reads the raw pairwise cosine-similarity scores generated by `03_embedding_extraction.ipynb` and evaluates recognition performance and demographic fairness across:

- **Models:** ArcFace, MobileFaceNet, MagFace
- **Codecs:** JPEG, JPEG XL, HEIC
- **Compression conditions:** L0–L5
- **Demographic groups:** Malay, Chinese, Indian

### Analysis pipeline

`raw_pairwise_scores.csv`  
→ validate and clean pairwise scores  
→ group-specific impostor Z-normalization  
→ calculate genuine/impostor statistics  
→ calculate FAR, FRR, and EER  
→ calculate Macro-Average EER and SER  
→ save **one final public result file**

### Final public output

```text
results/
└── data/
    └── fairness_summary_table.csv
```

All intermediate DataFrames remain in memory and are **not saved as separate CSV files**.

All numeric values in the final summary table are rounded to **2 decimal places**.

> **Research preservation:** The normalization method, EER threshold-selection logic, FAR/FRR calculations, demographic grouping, Macro-Average EER, and SER logic are preserved from the existing experiment.

> **Privacy:** `raw_pairwise_scores.csv` contains filename-derived identifiers and is treated as a private intermediate file. If Notebook 03 created it inside `results/data/`, this notebook moves it to `data/private_results/` before analysis so that the public results folder contains only the aggregate summary table.

## Imports

In [ ]:
from pathlib import Path
import shutil

import numpy as np
import pandas as pd

from sklearn.metrics import roc_curve

from scipy.interpolate import interp1d
from scipy.optimize import brentq

# Display numerical results consistently inside the notebook.
pd.options.display.float_format = "{:.2f}".format

## Project Paths

The repository separates:

- **public aggregate results** → `results/data/`
- **private intermediate pairwise scores** → `data/private_results/`

Notebook 03 may initially create `raw_pairwise_scores.csv` inside `results/data/`. When Notebook 04 starts, that file is moved into the private data area rather than deleted.

This keeps the public result directory suitable for GitHub while retaining the raw intermediate data locally.

In [ ]:
PUBLIC_RESULTS_DIR = Path("../results/data")
PRIVATE_RESULTS_DIR = Path("../data/private_results")

PUBLIC_RAW_SCORES_PATH = (
    PUBLIC_RESULTS_DIR / "raw_pairwise_scores.csv"
)

PRIVATE_RAW_SCORES_PATH = (
    PRIVATE_RESULTS_DIR / "raw_pairwise_scores.csv"
)

SUMMARY_OUTPUT_PATH = (
    PUBLIC_RESULTS_DIR / "fairness_summary_table.csv"
)

PUBLIC_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PRIVATE_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

TARGET_RACES = [
    "Malay",
    "Chinese",
    "Indian",
]

EPS = 1e-9

## Keep Raw Pairwise Scores Private

`raw_pairwise_scores.csv` is required to reproduce the fairness calculations, but it should not be part of the public GitHub result folder because it contains participant-level pair identifiers.

The following logic:

1. Uses the private copy if it already exists.
2. Otherwise moves the copy produced by Notebook 03 from `results/data/` to `data/private_results/`.
3. Raises an error if the raw score file cannot be found.

No participant-level score file is written back into the public results folder.

In [ ]:
if PRIVATE_RAW_SCORES_PATH.exists():
    RAW_SCORES_PATH = PRIVATE_RAW_SCORES_PATH

elif PUBLIC_RAW_SCORES_PATH.exists():
    print(
        "Moving raw_pairwise_scores.csv "
        "out of the public results folder..."
    )

    shutil.move(
        str(PUBLIC_RAW_SCORES_PATH),
        str(PRIVATE_RAW_SCORES_PATH),
    )

    RAW_SCORES_PATH = PRIVATE_RAW_SCORES_PATH

else:
    raise FileNotFoundError(
        "raw_pairwise_scores.csv was not found. "
        "Run 03_embedding_extraction.ipynb first."
    )

print(
    "Private raw-score input:",
    RAW_SCORES_PATH,
)

## Data Cleaning

The raw score table is standardized to the fields required for analysis:

```text
Model
Codec
Level
Race
MatchType
Score
```

Only:

- `genuine`
- `impostor`

pair types are retained.

The demographic scope is restricted to the three groups used in the study:

- Malay
- Chinese
- Indian

In [ ]:
def standardize_columns(df):
    """Standardize common column-name variants."""

    lower_to_original = {
        column.lower().strip(): column
        for column in df.columns
    }

    aliases = {
        "Model": [
            "model",
            "recognition_model",
        ],
        "Codec": [
            "codec",
            "format",
            "compression_format",
        ],
        "Level": [
            "level",
            "compression_level",
        ],
        "Race": [
            "race",
            "ethnicity",
            "group",
            "demographic_group",
        ],
        "MatchType": [
            "matchtype",
            "match_type",
            "pair_type",
            "type",
        ],
        "Score": [
            "score",
            "cosine_similarity",
            "cosine",
            "similarity",
            "raw_score",
        ],
    }

    rename_map = {}

    for standard_name, possible_names in aliases.items():
        for possible_name in possible_names:
            if possible_name in lower_to_original:
                rename_map[
                    lower_to_original[possible_name]
                ] = standard_name
                break

    df = df.rename(
        columns=rename_map
    ).copy()

    required_columns = [
        "Model",
        "Codec",
        "Level",
        "Race",
        "MatchType",
        "Score",
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            "Missing required columns: "
            f"{missing_columns}"
        )

    return df


def clean_raw_scores(df):
    """Clean and standardize the raw pairwise score table."""

    df = standardize_columns(df)

    df["Model"] = (
        df["Model"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    df["Codec"] = (
        df["Codec"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    df["Level"] = (
        df["Level"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    df["Race"] = (
        df["Race"]
        .astype(str)
        .str.strip()
        .str.title()
    )

    df["MatchType"] = (
        df["MatchType"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    df["Score"] = pd.to_numeric(
        df["Score"],
        errors="coerce",
    )

    df = df[
        df["MatchType"].isin(
            ["genuine", "impostor"]
        )
    ].copy()

    df = df.dropna(
        subset=["Score"]
    ).copy()

    df = df[
        df["Race"].isin(
            TARGET_RACES
        )
    ].copy()

    return df

## Evaluation Helper Functions

### Sample standard deviation

Within each demographic condition, score variability is calculated using sample standard deviation (`ddof=1`), matching the existing summary-table methodology.

### FAR, FRR, and EER

At a selected threshold:

$$
FAR =
\frac{\mathrm{Impostor\ scores\ accepted\ as\ genuine}}
{\mathrm{All\ impostor\ scores}}
$$

$$
FRR =
\frac{\mathrm{Genuine\ scores\ rejected\ as\ impostor}}
{\mathrm{All\ genuine\ scores}}
$$

$$
EER =
\frac{FAR + FRR}{2}
$$

The existing threshold procedure is preserved:

1. Build the ROC curve from genuine and impostor scores.
2. Find the threshold where `|FAR - FRR|` is smallest.
3. Recalculate FAR and FRR directly at that threshold.
4. Average the recalculated FAR and FRR to obtain EER.

### SER

For a model/codec/compression condition:

$$
SER =
\frac{\max(EER_g)}
{\min(EER_g)}
$$

SER is reported only when the minimum demographic EER is greater than zero. If any demographic group has zero EER, SER is left blank (`NaN`), matching the existing `fairness_summary_table.csv`.

In [ ]:
def sample_std(values):
    """Sample standard deviation used in the experiment."""

    values = np.asarray(
        values,
        dtype=float,
    )

    if len(values) <= 1:
        return 0.0

    return float(
        np.std(
            values,
            ddof=1,
        )
    )


def compute_eer_threshold(
    genuine_scores,
    impostor_scores,
):
    """
    Calculate EER and the corresponding interpolated threshold.

    Higher similarity scores indicate a more likely genuine match.

    The EER operating point is obtained by interpolating the ROC curve
    to find the point where FAR and FRR are equal.
    """

    genuine_scores = np.asarray(
        genuine_scores,
        dtype=float,
    )

    impostor_scores = np.asarray(
        impostor_scores,
        dtype=float,
    )

    if (
        len(genuine_scores) == 0
        or len(impostor_scores) == 0
    ):
        return {
            "EER": np.nan,
            "Threshold": np.nan,
            "FAR": np.nan,
            "FRR": np.nan,
        }

    y_true = np.concatenate(
        [
            np.ones(
                len(genuine_scores)
            ),
            np.zeros(
                len(impostor_scores)
            ),
        ]
    )

    y_scores = np.concatenate(
        [
            genuine_scores,
            impostor_scores,
        ]
    )

    far_curve, tpr, thresholds = roc_curve(
        y_true,
        y_scores,
    )

    frr_curve = 1 - tpr

    # --------------------------------------------------------
    # Find the interpolated FAR = FRR crossing
    # --------------------------------------------------------

    tpr_interpolator = interp1d(
        far_curve,
        tpr,
        kind="linear",
    )

    eer = brentq(
        lambda x:
            1.0
            - x
            - tpr_interpolator(x),
        0.0,
        1.0,
    )

    # Interpolate the threshold at the EER operating point.
    threshold_interpolator = interp1d(
        far_curve,
        thresholds,
        kind="linear",
    )

    threshold = float(
        threshold_interpolator(eer)
    )

    # At the interpolated EER operating point,
    # FAR and FRR are equal by definition.
    far = float(eer)
    frr = float(eer)

    return {
        "EER": float(eer),
        "Threshold": threshold,
        "FAR": far,
        "FRR": frr,
    }


def compute_ser(eer_values):
    """
    Calculate SER only when every demographic EER is
    greater than zero.

    If the minimum EER is zero, return NaN.
    """

    eer_values = pd.Series(
        eer_values,
        dtype=float,
    ).dropna()

    if eer_values.empty:
        return np.nan

    minimum_eer = eer_values.min()
    maximum_eer = eer_values.max()

    if minimum_eer <= 0:
        return np.nan

    return float(
        maximum_eer
        / minimum_eer
    )

## Load Raw Pairwise Scores

The raw file remains private and is loaded only as the analytical input for this notebook.

No cleaned or normalized participant-level CSV is exported.

In [ ]:
df_raw = pd.read_csv(
    RAW_SCORES_PATH
)

df_raw = clean_raw_scores(
    df_raw
)

print(
    f"Rows used: {len(df_raw):,}"
)

print(
    "Models:",
    sorted(
        df_raw["Model"].unique()
    ),
)

print(
    "Codecs:",
    sorted(
        df_raw["Codec"].unique()
    ),
)

print(
    "Levels:",
    sorted(
        df_raw["Level"].unique()
    ),
)

print(
    "Demographic groups:",
    sorted(
        df_raw["Race"].unique()
    ),
)

## Group-Specific Impostor Z-Normalization

Scores are normalized separately for every:

```text
Model × Codec × Level × Race
```

The impostor distribution provides the normalization reference:

$$
Z =
\frac{Score - \mu_{impostor}}
{\sigma_{impostor}}
$$

The same impostor mean and standard deviation are applied to both genuine and impostor scores within that condition.

If the impostor standard deviation is zero, `EPS = 1e-9` is used only as a numerical fallback.

In [ ]:
impostor_stats = (
    df_raw[
        df_raw["MatchType"]
        == "impostor"
    ]
    .groupby(
        [
            "Model",
            "Codec",
            "Level",
            "Race",
        ]
    )["Score"]
    .agg(
        Imp_Mean="mean",
        Imp_Std=lambda values: (
            sample_std(values)
        ),
    )
    .reset_index()
)

impostor_stats[
    "Imp_Std"
] = (
    impostor_stats["Imp_Std"]
    .replace(
        0,
        np.nan,
    )
    .fillna(
        EPS
    )
)

df_norm = df_raw.merge(
    impostor_stats,
    on=[
        "Model",
        "Codec",
        "Level",
        "Race",
    ],
    how="left",
)

df_norm["Z_Score"] = (
    df_norm["Score"]
    - df_norm["Imp_Mean"]
) / df_norm["Imp_Std"]

print(
    "✅ Group-specific impostor "
    "Z-normalization complete."
)

## Per-Demographic Recognition Metrics

For each:

```text
Model × Codec × Level × Race
```

the notebook calculates:

- raw genuine mean
- raw genuine standard deviation
- raw impostor mean
- raw impostor standard deviation
- normalized genuine mean
- normalized impostor mean
- FRR
- FAR
- EER
- normalized-score threshold

Raw similarity statistics and error rates are converted to percentages for the final table, matching the existing summary-table format.

In [ ]:
summary_records = []

for (
    model,
    codec,
    level,
    race,
), group_df in df_norm.groupby(
    [
        "Model",
        "Codec",
        "Level",
        "Race",
    ]
):
    genuine = group_df[
        group_df["MatchType"]
        == "genuine"
    ]

    impostor = group_df[
        group_df["MatchType"]
        == "impostor"
    ]

    genuine_raw = (
        genuine["Score"]
        .dropna()
        .values
    )

    impostor_raw = (
        impostor["Score"]
        .dropna()
        .values
    )

    genuine_norm = (
        genuine["Z_Score"]
        .dropna()
        .values
    )

    impostor_norm = (
        impostor["Z_Score"]
        .dropna()
        .values
    )

    threshold_metrics = (
        compute_eer_threshold(
            genuine_norm,
            impostor_norm,
        )
    )

    summary_records.append(
        {
            "Model": model,
            "Codec": codec,
            "Level": level,
            "Ethnic": race,

            "raw_genuine_mean (%)": (
                np.mean(genuine_raw)
                * 100
            ),

            "std_genuine (percentage points)": (
                sample_std(
                    genuine_raw
                )
                * 100
            ),

            "raw_impostor_mean (%)": (
                np.mean(impostor_raw)
                * 100
            ),

            "std_impostor (percentage points)": (
                sample_std(
                    impostor_raw
                )
                * 100
            ),

            "norm_genuine_mean": (
                np.mean(
                    genuine_norm
                )
            ),

            "norm_impostor_mean": (
                np.mean(
                    impostor_norm
                )
            ),

            "FRR (%)": (
                threshold_metrics[
                    "FRR"
                ]
                * 100
            ),

            "FAR (%)": (
                threshold_metrics[
                    "FAR"
                ]
                * 100
            ),

            "EER (%)": (
                threshold_metrics[
                    "EER"
                ]
                * 100
            ),

            "Threshold (normalized score)": (
                threshold_metrics[
                    "Threshold"
                ]
            ),
        }
    )

df_summary = pd.DataFrame(
    summary_records
)

print(
    "✅ Per-demographic recognition "
    "metrics calculated."
)

## Macro-Average EER and SER

Fairness is summarized across Malay, Chinese, and Indian groups for each:

```text
Model × Codec × Level
```

### Macro-Average EER

$$
Macro\text{-}Average\ EER
=
\frac{1}{G}
\sum_{g=1}^{G} EER_g
$$

where \(G\) is the number of demographic groups.

### SER

$$
SER =
\frac{\max(EER_g)}
{\min(EER_g)}
$$

The same Macro-Average EER and SER values are attached to each demographic row belonging to the corresponding model/codec/level condition. This makes the single summary CSV convenient for Streamlit, Power BI, and other visualization tools.

In [ ]:
fairness_records = []

for (
    model,
    codec,
    level,
), group_df in df_summary.groupby(
    [
        "Model",
        "Codec",
        "Level",
    ]
):
    eer_values = (
        group_df["EER (%)"]
        .dropna()
    )

    macro_average_eer = (
        eer_values.mean()
        if not eer_values.empty
        else np.nan
    )

    ser = compute_ser(
        eer_values
    )

    fairness_records.append(
        {
            "Model": model,
            "Codec": codec,
            "Level": level,
            "Macro-Average EER (%)": (
                macro_average_eer
            ),
            "SER": ser,
        }
    )

df_fairness = pd.DataFrame(
    fairness_records
)

df_summary = df_summary.merge(
    df_fairness,
    on=[
        "Model",
        "Codec",
        "Level",
    ],
    how="left",
)

print(
    "✅ Macro-Average EER and "
    "SER calculated."
)

## Final Fairness Summary Table

This is the **only result table saved by Notebook 04**.

The columns match the existing portfolio summary structure:

```text
Model
Codec
Level
Ethnic
raw_genuine_mean (%)
std_genuine (percentage points)
raw_impostor_mean (%)
std_impostor (percentage points)
norm_genuine_mean
norm_impostor_mean
FRR (%)
FAR (%)
EER (%)
Threshold (normalized score)
Macro-Average EER (%)
SER
```

All numerical values are rounded to **2 decimal places** before export.

SER remains blank when the minimum demographic EER is zero because the ratio is undefined in that condition.

In [ ]:
# Sort conditions for easier inspection and visualization.
model_order = {
    "arcface": 0,
    "magface": 1,
    "mobilefacenet": 2,
}

codec_order = {
    "NONE": 0,
    "JPEG": 1,
    "JXL": 2,
    "HEIC": 3,
}

race_order = {
    "Malay": 0,
    "Chinese": 1,
    "Indian": 2,
}


def level_number(level):
    """Return the numeric part of L0-L5."""

    try:
        return int(
            str(level)
            .upper()
            .replace("L", "")
        )
    except ValueError:
        return 999


df_summary["_model_order"] = (
    df_summary["Model"]
    .map(model_order)
    .fillna(999)
)

df_summary["_codec_order"] = (
    df_summary["Codec"]
    .map(codec_order)
    .fillna(999)
)

df_summary["_level_order"] = (
    df_summary["Level"]
    .map(level_number)
)

df_summary["_race_order"] = (
    df_summary["Ethnic"]
    .map(race_order)
    .fillna(999)
)

df_summary = (
    df_summary
    .sort_values(
        [
            "_model_order",
            "_codec_order",
            "_level_order",
            "_race_order",
        ]
    )
    .drop(
        columns=[
            "_model_order",
            "_codec_order",
            "_level_order",
            "_race_order",
        ]
    )
    .reset_index(
        drop=True
    )
)

numeric_columns = (
    df_summary
    .select_dtypes(
        include=[
            np.number
        ]
    )
    .columns
)

# Remove negative zero values before final rounding.
for column in numeric_columns:
    df_summary[column] = (
        df_summary[column]
        .mask(
            df_summary[column].abs()
            < 0.005,
            0.0,
        )
        .round(2)
    )

# Save the only public result CSV.
df_summary.to_csv(
    SUMMARY_OUTPUT_PATH,
    index=False,
    float_format="%.2f",
)

print(
    "✅ Final fairness summary saved:"
)

print(
    SUMMARY_OUTPUT_PATH
)

print(
    f"Rows: {len(df_summary)}"
)

## Clean Legacy Notebook 04 Outputs

Earlier development versions of Notebook 04 created several intermediate CSV files and a `fairness_analysis_outputs/` directory.

They are no longer required because the public portfolio uses only `fairness_summary_table.csv`.

This cleanup removes only the known legacy outputs created by the earlier Notebook 04 workflow. It does not remove the private raw score file.

In [ ]:
legacy_files = [
    PUBLIC_RESULTS_DIR
    / "fairness_metrics_final.csv",
]

legacy_directories = [
    PUBLIC_RESULTS_DIR
    / "fairness_analysis_outputs",
]

for legacy_file in legacy_files:
    if legacy_file.exists():
        legacy_file.unlink()

for legacy_directory in legacy_directories:
    if legacy_directory.exists():
        shutil.rmtree(
            legacy_directory
        )

print(
    "✅ Legacy Notebook 04 outputs removed."
)

## Final Output Check

After Notebook 04 completes, the intended **public** result folder is:

```text
results/
└── data/
    └── fairness_summary_table.csv
```

The private pairwise input is retained separately:

```text
data/
└── private_results/
    └── raw_pairwise_scores.csv
```

This separation supports:

- reproducibility during local development
- participant-data privacy
- a clean GitHub repository
- direct visualization from one aggregate CSV using Streamlit or Power BI

In [ ]:
public_items = sorted(
    item.name
    for item in PUBLIC_RESULTS_DIR.iterdir()
)

print(
    "Public results folder:"
)

for item in public_items:
    print(
        f" - {item}"
    )

unexpected_items = [
    item
    for item in public_items
    if item
    != "fairness_summary_table.csv"
]

if unexpected_items:
    print(
        "\n⚠️ Additional files are still present:",
        unexpected_items,
    )
else:
    print(
        "\n✅ Public results folder contains "
        "only fairness_summary_table.csv"
    )

## 13. Preview

The final aggregate result table can now be used directly for:

- Streamlit dashboards
- Power BI dashboards
- portfolio figures
- README result visualizations

The participant-level pairwise score file is not required for those public-facing visualizations.

In [ ]:
display(
    df_summary.head(
        15
    )
)

## Output of This Stage

The final pipeline is:

```text
01 — Facial Image Preprocessing
        ↓
02 — Image Compression
        ↓
03 — Pretrained Face Embedding Extraction
     + Raw Pairwise Similarity Scores
        ↓
04 — Recognition and Fairness Evaluation
        ↓
results/data/fairness_summary_table.csv
```

The final public CSV contains aggregate recognition and fairness metrics at the:

```text
Model × Codec × Level × Demographic Group
```

granularity.

This is sufficient for the planned portfolio visualizations while avoiding public distribution of participant-level raw comparison records.